In [1]:
import pandas as pd
from decons_tg import *
from telethon.tl.types import PeerChannel

In [22]:
api_id = 35765655 # make sure you enter this as an integer, not a string
api_hash = "976c35d49039db41f30304d07db0e980" # hash string from API
phone = "+447418377418" #phone number

client = TelegramClient("mySession", api_id, api_hash)

await client.connect()
if not await client.is_user_authorized():
    await client.send_code_request(phone)
    await client.sign_in(phone, input('Enter the code: '))

In [2]:
df = pd.read_parquet("/Users/rubenvandijkhuizen/Downloads/nl_50_nn_bertje_leiden_labeled.pqt" ,engine="fastparquet")

In [26]:
def generate_link_graph(ldf, scrape_name, PlatformUserCache):
    N = load_cache(PlatformUserCache)
    
    url_df = ldf[ldf["domain"].isin(["t.me","www.youtube.com","youtube.com","youtu.be"]) == False].copy()
    cp_nodes = dict()
    cp_edgelist = list()
    for row in url_df.itertuples():
        dt = platform_url_handler(row.domain, row.full_url, N)
        key = dt["channel_type"] + "@" + dt["username"]
        if key in cp_nodes.keys():
            edge = (scrape_name, key)
            cp_edgelist.append(edge)
        else:
            cp_nodes[key] = dt
            edge = (scrape_name, key, row.message_date)
            cp_edgelist.append(edge)
    return cp_nodes, cp_edgelist

async def fetch_title(id):
    channel = await client.get_entity(PeerChannel(id))

    return channel.username


In [29]:
# 0 = Covid / Vaccines
# 3 = Russia/Ukraine War
# 12 = Climate / Chemtrails
# 19 = Farmers
# 21 = AI, Radiation and Viruses


# Which topics to include
topics = [10]

peer_id_lookup = dict()

# Loop over al peer_ids and create a lookup table
for topic in topics:

    topicDf = df[df["topic_label_leiden"] == topic]
    id_counts = topicDf["peer_id"].value_counts()

    # Loop over all peer_ids
    for peer_id in id_counts.index:
        
        if peer_id in peer_id_lookup.keys():
            username = peer_id_lookup[peer_id]
        else:
            username = await fetch_title(peer_id)
            username = username.lower()
            peer_id_lookup[peer_id] = username


        




In [30]:
for topic in topics:
    tDf = df[df["topic_label_leiden"] == topic]
    tLDf = prepare_url_dataframe(tDf)

    id_counts = tLDf["peer_id"].value_counts()

    topicNodes = dict()
    topicEdges = list()

    print(len(id_counts.index))

    for peer_id in id_counts.index:

        subLDf = tLDf[tLDf["peer_id"] == peer_id]
        nodelist, edgelist = generate_link_graph(subLDf, peer_id_lookup[peer_id], PlatformUserCache)
        topicNodes = topicNodes | nodelist
        topicEdges = topicEdges + edgelist

    topicNodelist = generate_url_share_nodes(topicNodes)
    topicEdgeList = generate_url_share_edgelist(topicEdges)
    topicEdgeList["source"] = topicEdgeList["source"].apply(lambda x: "Telegram@" + x)
    topicNodelist.to_csv("New Data/Topic_" + str(topic) + "_url_share_nodes.csv", index = False)
    topicEdgeList.to_csv("New Data/Topic_" + str(topic) + "_url_share_edgelist.csv", index = False)  
    tLDf.to_csv("New Data/Topic_" + str(topic) + "_URL_DataFrame.csv")



    

Checking for and expanding shortened URLs...


100%|██████████| 13610/13610 [01:07<00:00, 201.12it/s]


Extracting domains...


100%|██████████| 13610/13610 [00:00<00:00, 166105.01it/s]


45


In [ ]:
for topic in topics:
    tDf = df[df["topic_label_leiden"] == topic]

    print(len(tDf))

    tDf.to_csv("New Data/Topic_" + str(topic) + "_Global_DataFrame.csv")

    

36833
